<a href="https://colab.research.google.com/github/IFuentesSR/yield_modelling/blob/main/sampling_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee
ee.Authenticate()
ee.Initialize(project='xxxx') # Use your own google cloud project

In [2]:
import folium
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print(folium.__version__)

0.20.0


In [3]:
table = ee.FeatureCollection("projects/upbeat-imprint-269809/assets/bounds_yield")
image = ee.Image("projects/upbeat-imprint-269809/assets/yields_aggregated")
dem = ee.Image("USGS/SRTMGL1_003")
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
ERA5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")

In [4]:
def era_offset(img):
    return img.multiply(1000).multiply(-1).copyProperties(img, img.propertyNames())

ERA5 = ERA5.filterDate('2018-03-01', '2018-08-15').select('potential_evaporation_sum')
ERA5 = ERA5.map(era_offset)


In [5]:
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

In [6]:
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12',
             'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB',
             'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE',
             'probability', 'dark_pixels', 'cloud_transform', 'shadows', 'NDVI', 'VREI', 'NDRE']

In [10]:
cloudProbabilityThreshold = 40


def getS2_CLOUD_PROBABILITY(geo):
    innerJoined = ee.Join.inner().apply(primary=ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(geo).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)),
                                        secondary=ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(geo),
                                        condition=ee.Filter.equals(leftField='system:index',
                                                                   rightField='system:index'))
    def mergeImageBands(joinResult):
        return ee.Image(joinResult.get('primary')).addBands(joinResult.get('secondary'))

    newCollection = innerJoined.map(mergeImageBands)
    return ee.ImageCollection(newCollection)


def maskClouds(_img):
    props = _img.propertyNames()
    cloudMask = _img.select('probability').gt(cloudProbabilityThreshold)
    return _img.addBands(cloudMask.rename('clouds')).copyProperties(_img, props)


def add_shadow_bands(img):
    not_water = img.select('SCL').neq(6)
    SR_BAND_SCALE = 1e4
    CLD_PRJ_DIST = 1
    NIR_DRK_THRESH = 0.15
    clouds = ee.Image(maskClouds(img)).select('clouds')
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).rename('dark_pixels')
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    cld_proj = (clouds.directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(crs=img.select(0).projection(), scale=100)
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

def masking(img):
    return img.updateMask(img.select('probability').gt(cloudProbabilityThreshold).Not()).updateMask(img.select('shadows').Not())


def addNDWI(img):
  bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
  img2 = img.select(bands).divide(10000)
  return img2.addBands(img2.normalizedDifference(['B8', 'B4']).rename('NDVI')) \
  .addBands(img2.normalizedDifference(['B3', 'B8']).rename('NDWI_mcFeeters')) \
  .addBands(img2.normalizedDifference(['B8', 'B5']).rename('NDRE')).copyProperties(img, img.propertyNames())



In [7]:
def buffering(fea):
    props = fea.propertyNames()
    geom = fea.geometry().buffer(-30)
    return ee.Feature(geom).copyProperties(fea, props)

feas = table.map(buffering)


In [13]:
def sampling(fea):
    id = fea.get('id')
    s2 = getS2_CLOUD_PROBABILITY(fea.geometry()).filterDate('2018-03-01', '2018-08-15')
    s2_shdw = s2.map(add_shadow_bands)
    s2_masked = s2_shdw.map(masking)
    NDWI_coll = s2_masked.map(addNDWI)
    area = fea.geometry().area()
    def inner(img):
        date = ee.Date(img.get('system:time_start'))
        doy = date.format('D')
        rain = chirps.filterDate('2018-03-01', date).reduce('sum')
        pet = ERA5.filterDate('2018-03-01', date).reduce('sum')
        date = date.format('YYYY-MM-dd')
        img = img.addBands(image.rename('yield')).addBands(slope).addBands(aspect).addBands(dem) \
        .addBands(ee.Image.pixelLonLat()) \
        .addBands(rain.rename('rain')).addBands(pet.rename('pet'))
        img = img.unmask(ee.Image([-999]))
        def inner_drop(i):
            return i.setMulti({'drop': 1, 'date':date, 'id':id, 'doy':doy})
        def inner_keep(i):
            return i.setMulti({'drop': 0, 'date':date, 'id':id, 'doy':doy})
        img_area = img.select('NDVI').gt(-999).multiply(ee.Image.pixelArea()).reduceRegion('sum', fea.geometry(), 20).values().get(0)
        condition = ee.Algorithms.If(ee.Number(img_area).divide(area).lt(0.7),
                                     ee.FeatureCollection([ee.Feature(None, {})]).map(inner_drop),
                                     ee.FeatureCollection(img.sample(fea.geometry(), 20)).map(inner_keep))
        return condition
    return NDWI_coll.map(inner).flatten()

samples = feas.limit(2).map(sampling)

ee.batch.Export.table.toDrive(samples.flatten().filter(ee.Filter.eq('drop', 0)), 'testing').start()
